# Prerrequisitos

In [2]:
using LinearAlgebra

# Reflexiones de Householder

Dado $v\in\mathbb{R}^m$ defina la matrix de Housholder por $\displaystyle H_v=I - 2\frac{vv^T}{v^Tv}.$ Decimos que $v$ es el vector de Hoseholder.  Podemos verificar que 

1. $H$ es una isometría en la norma $\|\cdot\|_2$.
2. $H$ es simétrica 
3. $H$ es ortogonal

Dado $x\in \mathbb{R}^m$, la reflexión de $x$ con eje de reflexión el huperplano $v^\perp$ es dada por 
$$
y=H_vx = x- 2\frac{v^Tx}{v^Tv}v.
$$
Note que para cacular $y$ no es necesario calcular todas la entradas de la matriz $H$, es suficiente calcular el producto interno $s=v^Tx$ y realizar la resta $x-\beta s v$ donde $\beta= 2\frac{1}{v^Tv}$. El valor de $\beta$ puede ser precalculado. 

Dados $x,y\in \mathbb{R}^m$. ¿Existe $v$ tal que $H_vx=y$?. Si $\|x\|_2=\|y\|_2$ la respuesta es afirmativa  y 
$v=\alpha (x-y)$. 

Para usar Householder en ortogonalización procedemos como sigue. Suponga que queremos orgonaliar las columnas de $A=(a_1,a_2,\dots,a_n)\in \mathbb{R}^{m\times n}$. Considere inicialmente el caso de columnas linealmente independientes. Iniciamos la ortogonalización de Householder calculando $v$ que transforme $a_1$ en un multiplo del primer vector canónico $e_1=(1,0,\dots,0)$. Es decir $H_v a_1= \sigma e_1$ con $\sigma = \pm \|x\|_2$.



Suponga que $x\in\mathbb{R}^m$ y queremos calcular $v$ tal que $H_v x$ sea múltiplo de $e_1$, digamos $\sigma e_1$. Debe ser $\sigma= \pm\|x\|_2$. Observe que 
$$
x-\sigma e_1 = (x_1-\sigma,x_2,x_3,\dots,x_m).
$$
En la resta $x_1-\sigma$ puede ocurrir cancelacion catastrófica. Tenemos las siguientes opciones para evitar la cancelación catastrófica, 

*Opción 1:* Observe que si $x_1>0$ entonces $x$ esta mas cerca a  $\|x\|_2e_1$ que a $-\|x\|_2e_1$. Entonces podemos tomar $\sigma = -\mbox{sign}(x_1)$ y evitamos la cancelación catastrófica. 
Despues de calcular $v_1=x_1-\sigma$ podemos calcular 
$$
\beta =  \frac{2}{v^Tv}= \frac{2}{ v^Tv} = \frac{2}{ v_1^2+s}
$$
donde $s=x_2^2+\dots+x_m^2$.

*Opción 2:* Podemos seleccionar $\sigma=\|x\|_2$ pero tener cuidado al calcular $x_1-\sigma$. 
- Si $x_1<0$ entonces podemos calcular $v_1=x_1-\sigma$ (no hay resta).
- Si $x_1>0$ entonces calculamos 
    $$
    x_1-\sigma = (x_1-\sigma) \frac{x_1+\sigma}{x_1+\sigma}= \frac{x_1^2-\|x\|_2^2}{x_1+\sigma}= 
    -\frac{x_2^2+\dots+x_m^2}{x_1+\sigma} = - \frac{s}{x_1+\sigma}
    $$
    donde $s=x_2^2+\dots+x_m^2$.
    
    Note tambien que se puede calcular $v^Tv$ como sigue (recuerde que $\sigma=\|x\|_2$)
    $$
    v^Tv= (x_1-\sigma)^2+x_2^2+\dots+x_m^2 = x_1^2-2\sigma x_1+\sigma^2+x_2^2+\dots+x_m^2= 
    2\sigma^2 - 2\sigma x_1= 2\sigma (\sigma -x)= -2\sigma v_1.
    $$

En muchas implementaciones para ahorar en memoria se normaliza $v$ de tal forma que $\tilde{v}=v/v_1$ y se guardan las entradas $n-1$ últimas entradas de  $v_2/v_1,\dots,v_m/v_1$  en las entradas de $x_2,x_3, \dots,x_m$ (debajo de la diagonal de la matriz $R$ en aplicacion de ortonormalización).

Tenemos entonces el siguiente algorimos que dado un vector $x$, calcula de forma correcta el vector de Householder $v$. Algoritmo 5.1.1, página 210 del texto guía. 

In [3]:
function house(x)
    y = copy(x)
    n = size(x,1) #(m,n)
    s = x[2:n]'x[2:n]
    v = [1; x[2:n]]
    
    if s == 0
        β = 0
        
    else
        mu = sqrt(x[1]^2 + s)
        if x[1] <= 0
            v[1] = x[1] - mu
        else
            v[1] = -s /(x[1]+mu)
        end
        β = 2((v[1])^2)/(s + (v[1])^2)
        v=v/v[1]
    end
    return v, β
end


house (generic function with 1 method)

Considere el siguiente ejemplo. 

In [3]:
x=[0.5;sqrt(3)/2;4];
v, β= house(x)
println(v)
println(β)

[1.0, -0.23902847260678473, -1.104025224028098]
0.8787321874818336


In [4]:
H=UniformScaling(1)- 2(v*v')/(v'v)

3×3 Matrix{Float64}:
 0.121268   0.210042   0.970143
 0.210042   0.949794  -0.231892
 0.970143  -0.231892  -0.0710618

In [5]:
H*x

3-element Vector{Float64}:
 4.123105625617661
 1.1102230246251565e-16
 5.551115123125783e-17

Note que si estamos haciendo la factorización de una sola columna, $x$, entonces $H^T$ corresponde a la matriz $Q$ de la factorización $QR$, **pero esta vez la matriz $Q$ es cuadrada y genera todo $\mathbb{R}^m$**. Es decir, obtenemos la factorización completa. En esta caso las filas $2:m$ de la matriz $H$ son vectores ortogonales a $x$. Recuerde que si queremos obtener la factorizació completa con GS tendríamos que iniciar con $m$ columnas, es decir, tendriamos que encontrar $m-n$ columnas linealmente independientes a las que tenemos y despues aplicar GS. 

Consideremos nuevamente la matrix $A=(a_1,a_2,\dots,a_n)$. Aplicando el procedimiento anterior construimos $H_1$ tal que 
$$
A_1=H_1A = (\sigma e_1, H_1a_2,\dots, H_1a_n)
$$
es una matriz con $A_1(2:m,1)=(0,\dots,0)$. 
Ahora debemos obtener entradas nulas también en la segunda columna debano de la diagonal.
Para continuar con la triagularización de Householder construimos una reflexion de Householder en $\mathbb{R}^{m-1}$ que transforme el vector $A_1(2:m,\color{red}{2})$ en el vector $(1,0,\dots,0)\in \mathbb{R}^{m-1}$. Sea $\tilde{H}_2$ esta transformación. Defina 
$$
H_2= \begin{pmatrix}1 & 0\\ 0 &\tilde{H}_2\end{pmatrix}.
$$
Vemos que $H_2H_1a_2= (A_1(1,2), \sigma_2,0,\dots,0)\in \mathbb{R}^m$. Concluimos que 
$$
A_2= H_2H_1 A =  ( \sigma e_1, H_2H_1a_2, H_2H_1a_3,\dots, H_2H_1a_n)  
$$
es una matriz con entradas nulas debajo de la diagonal en las dos primeras columnas. 

Podemos continuar este proceso hasta la ultima columna y obtenemos una matriz triangular superior. Es decir, obtenemos 
$$
H_nH_{n-1}\cdots  H_1  A = R
$$
y por tanto $ A= H_1^TH_2^T\cdots H_n^T R =QR$ donde $Q=H_1H_2\cdots H_n$. 

Note que quedamos con la matriz $R$ pero para poder obtener la matriz $Q$ debemos, en prinicipio, multiplicar las matrices $H_i$, lo cual es computacionalmente costoso. Veamos primero como queda el algoritmos para obtener $R$ y después  mostraremos un algoritmo para construir $Q$. 


Considere el algorimos en la página 211 que usa el algoritmo anterior para obtener la matriz $R$ de la factorizacion $QR$. Recuerde que asumimos que las columnas de $A$ son linealmente independientes. 

In [4]:
function Rhouse(B)
    A=copy(B)
    m,n = size(A)
    for j = 1:n
        v, β = house(A[j:m,j])
        A[j:m,j:n] = ( UniformScaling(1) - β*v*v')A[j:m,j:n]
#      A[j+1:m,j]=v[2:end]
    end
    return A
end

Rhouse (generic function with 1 method)

In [9]:
n = 3 ;
m = 5 ;
A = rand(m,n)

5×3 Matrix{Float64}:
 0.221639  0.796929   0.25159
 0.757768  0.458052   0.825582
 0.643566  0.0341317  0.466831
 0.912214  0.787216   0.554411
 0.592682  0.990804   0.276661

In [10]:
R=Rhouse(A)

5×3 Matrix{Float64}:
 1.49027       1.24208       1.10819
 2.77556e-17   0.951168     -0.0749697
 1.249e-16     5.55112e-17   0.336178
 5.55112e-17  -1.11022e-16   4.16334e-17
 0.0          -2.22045e-16   4.16334e-17

# Construcción de matriz $Q$

Algoritmo de acumulación progresiva

In [5]:
function QFA(B)
    A = copy(B)
    Q = UniformScaling(1)
    m,n = size(A)
    for j = 1:n
        v,β = house(A[j:m,j])
        v1 = zeros(j-1,1)
        v = [v1;v]
        Qj = UniformScaling(1)-β*v*v'
        Q = Q*Qj
        A=Qj*A;
    end
    return Q, A
end

QFA (generic function with 1 method)

In [38]:
Q,R=QFA(A)
Q'*Q
A-Q*R

5×3 Matrix{Float64}:
 -1.11022e-16   0.0          -1.11022e-16
  0.0           0.0          -1.66533e-16
 -5.55112e-17  -8.32667e-17   1.11022e-16
  0.0           2.77556e-17   0.0
  0.0           8.32667e-17   2.22045e-16

In [40]:
Q[:,1:n]

5×3 Matrix{Float64}:
 0.614812    0.68607    -0.282268
 0.212       0.137376    0.161451
 0.329937    0.0144081   0.209613
 0.0635428   0.195369    0.921972
 0.681298   -0.687065    0.016983

In [41]:
Q[:,n+1:m]

5×2 Matrix{Float64}:
 -0.217668    0.155753
  0.939795    0.164017
  0.0478576  -0.919079
 -0.240311    0.223633
 -0.0967741   0.23264

In [42]:
n = 30 ;
m = 50 ;
A = rand(m,n);
Q,R=QFA(A);

In [44]:
opnorm(UniformScaling(1)-Q'Q)

4.328659056654992e-15

In [45]:
opnorm(A-Q*R)

6.92890433883397e-15

Mencionamos antes que realizar la multiplicación de las matrices de Householder puede ser costoso. En su lugar, cuando se requiere la $R$ podemos usar al siguiente resulado. 

**Teorema (Representación por bloques de Householder)** Si definimos 
$$
Q_1=H_1, \quad Q_2=H_1H_2, \dots, Q_i=H_1H_{2}\cdots H_{i-1}H_i= Q_{i-1}H_i
$$
entonces $Q_i=I +W_i Y_i^T$, $i=1,2,\dots,n$, $W_i, Y_i \in \mathbb{R}^{m\times i}$.

Algoritmo de acumulación regresiva

In [6]:
function VβHouse(A)
    m,n = size(A)
    β   = zeros(n,1)
    Y   = zeros(m,n)

    Am  = copy(A)
    
    # Primera actualización
    v1,β[1] = house(Am[:,1])
    Y[:,1]  = v1
    W       = -β[1]*v1
    Am      = ( UniformScaling(1) - β[1]*v1*v1')Am
    
    #De ahi para adelante
    for j = 2:n
        v1,β[j] = house(Am[j:m,j])
        Y[:,j]  = [ zeros(j-1,1) ; v1 ] 
        z       = -β[j]*(UniformScaling(1)+W*Y[:,1:j-1]')*Y[:,j]
        W       = [ W z ]
        Am[j:m,j:n] = ( UniformScaling(1) - β[j]*v1*v1')Am[j:m,j:n]
    end
    
    
    return Y,W,Am
end

VβHouse (generic function with 1 method)

In [38]:
Y,W,R = VβHouse(A)
Y


5×3 Matrix{Float64}:
  1.0        0.0        0.0
 -0.623749   1.0        0.0
 -1.22921    1.29209    1.0
 -0.476068  -0.602918  -1.75113
 -1.11093    0.121383   1.01955

In [73]:
W

5×3 Matrix{Float64}:
 -0.997017   -0.00194339  -0.590598
  0.0477908  -0.32859     -0.118235
  0.761254    0.466734    -0.132002
  0.088004    0.162531     1.09555
  0.640667   -0.552389     0.0179303

In [19]:
Q = UniformScaling(1) + W*Y'

5×5 Matrix{Float64}:
 0.445688  -0.256745  -0.0403942  -0.0632655   0.854289
 0.567844  -0.30876   -0.414463   -0.462888   -0.442918
 0.414015   0.798308   0.305457   -0.312667    0.0152151
 0.348276   0.298398  -0.498421    0.733132   -0.0612924
 0.431523  -0.335279   0.69632     0.38274    -0.264622

In [20]:
R

5×3 Matrix{Float64}:
  1.43765       1.26638       1.24157
 -1.66533e-16   0.32405       0.170211
 -2.77556e-17  -6.93889e-18   0.629568
 -5.55112e-17  -1.73472e-18   0.0
 -1.11022e-16   2.77556e-17  -5.55112e-17

In [80]:
A-Q*R

5×3 Matrix{Float64}:
 -9.02056e-17  -3.46945e-17   5.55112e-17
  6.93889e-17   0.0           0.0
  0.0           0.0           1.11022e-16
 -2.77556e-17  -8.32667e-17  -2.22045e-16
  1.11022e-16  -9.02056e-17   2.22045e-16

# Factorización QR por Householder por Bloques .
Para implementar este algoritmo, debemos implementar el algoritmo 5.2.1 del libro, donde básicamente es el algoritmo Rhouse de este documento, agregando dos líneas de código, este algoritmo encuentra las matrices de householder $H_1,\dots ,H_n$ tales que $Q=H_1\cdots H_n$ donde $Q^TA=R$.

In [44]:
function R1house(B)
    A=copy(B)
    m,n = size(A)
    for j = 1:n
        v, β = house(A[j:m,j])
        A[j:m,j:n] = ( UniformScaling(1) - β*v*v')A[j:m,j:n]
        if j<m
        A[j+1:m,j]=v[2:end]
        end
    end
    return A
end

R1house (generic function with 1 method)

In [9]:
n = 3 ;
m = 5 ;
A = rand(m,n)
R1house(A)

5×3 Matrix{Float64}:
  1.26956    1.27495     1.31939
 -0.609616   0.698372    0.952147
 -0.232367  -0.628247    0.235512
 -0.50994   -0.362636   -2.11642
 -1.0542    -0.0684582   1.05153

Ahora, podemos reorganizar y utilizar el algoritmo de householder por bloques.

In [47]:
function BHQRF(A)
    m,n = size(A)
    λ=1
    k=0
    r=0
    Am = copy(A)
    while λ <= n
        r = min(λ+r-1,n)
        k = k+1
        R1house(Am[λ:m,λ:n])
        Yk,Wk,Rk = VβHouse(Am[λ:m,λ:n])
        Qk = UniformScaling(1) + Wk*Yk'
        Am[λ:m, r+1:n] = (UniformScaling(1) + Wk*Yk')'*Am[λ:m,r+1:n]
        return Qk, Am[λ:m, r+1:n]
        λ = r + 1
    end
    
end
        
        
        
        
    

BHQRF (generic function with 1 method)

In [48]:
n = 3 ;
m = 5 ;
A = rand(m,n)
Q, R = BHQRF(A)
Q

5×5 Matrix{Float64}:
 0.433774   -0.258065    0.624925   0.331455   0.494822
 0.596907   -0.293353   -0.262815  -0.689079   0.117237
 0.519005    0.526159   -0.490814   0.437648   0.14614
 0.428322   -0.0590097   0.2702     0.148945  -0.847269
 0.0521145   0.753008    0.47591   -0.448985   0.0467432

In [50]:
Q*Q'

5×5 Matrix{Float64}:
  1.0           9.71445e-17   1.80411e-16   5.55112e-17  -6.93889e-17
  9.71445e-17   1.0          -1.8735e-16    1.52656e-16   1.47451e-17
  1.80411e-16  -1.8735e-16    1.0          -4.57967e-16  -2.92301e-16
  5.55112e-17   1.52656e-16  -4.57967e-16   1.0           2.22045e-16
 -6.93889e-17   1.47451e-17  -2.92301e-16   2.22045e-16   1.0

In [42]:
R

5×3 Matrix{Float64}:
  0.785406      0.43278       0.968342
  2.42861e-17   0.957425      0.443982
  5.72459e-17   3.05311e-16   0.545007
 -1.31839e-16  -2.77556e-16  -2.77556e-17
 -1.11022e-16  -2.77556e-16  -1.38778e-16

In [43]:
A-Q*R

5×3 Matrix{Float64}:
  5.55112e-17   0.0           8.32667e-17
  0.0           1.94289e-16   0.0
 -5.55112e-17  -2.77556e-16  -3.33067e-16
 -8.32667e-17  -4.44089e-16  -4.44089e-16
 -2.77556e-17  -2.77556e-16  -1.66533e-16